In [1]:
!pip install --upgrade pip
!pip install joblib scikit-learn pandas numpy
!pip install sentence-transformers torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install groq python-dotenv
print("✅ All packages installed successfully!")

✅ All packages installed successfully!


In [3]:
import joblib
import sklearn
print("✅ joblib version:", joblib.__version__)
print("✅ scikit-learn version:", sklearn.__version__)
print("Everything is now working!")

ModuleNotFoundError: No module named 'joblib'

In [2]:
import pandas as pd
import numpy as np
import json
import joblib
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import os
print("✅ All libraries loaded")

ModuleNotFoundError: No module named 'joblib'

In [ ]:
# Load ingredient database
db = pd.read_csv('data/ingredients_database.csv', encoding='latin-1')
print(f"Loaded {len(db)} ingredients")

# Use the same embedding model everywhere
emb_model = SentenceTransformer('all-MiniLM-L6-v2')  # free, runs on CPU

# Create embeddings for all ingredients + aliases
ingredient_texts = db['ingredient_name'].tolist()
if 'common_aliases' in db.columns:
    aliases = db['common_aliases'].fillna('').str.split(',').explode().str.strip().tolist()
    ingredient_texts.extend([a for a in aliases if a])

embeddings = emb_model.encode(ingredient_texts, batch_size=32, show_progress_bar=True)

# Train Random Forest
X_train, X_test, y_train, y_test = train_test_split(
    embeddings, db['concern_level'], test_size=0.2, random_state=42, stratify=db['concern_level']
)

clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
print("✅ Random Forest Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Save models
os.makedirs('models', exist_ok=True)
joblib.dump(clf, 'models/safety_rf_model.pkl')
joblib.dump(emb_model, 'models/embedding_model.pkl')
print("✅ Models saved to models/")

In [ ]:
# Load products.json
with open('data/products.json', 'r', encoding='utf-8') as f:
    PRODUCTS = json.load(f)

# Flatten all products into a single list with rich text
all_products = []
product_metadata = []
for category, prods in PRODUCTS.items():
    for p in prods:
        text = f"{p['name']} {p.get('brand','')} {p.get('key_ingredients','')} {' '.join(p.get('concerns',[]))} {p.get('category','')}"
        all_products.append(text)
        product_metadata.append(p)

product_embeddings = emb_model.encode(all_products, batch_size=32, show_progress_bar=True)
np.save('models/product_embeddings.npy', product_embeddings)
joblib.dump(product_metadata, 'models/product_metadata.pkl')
print(f"✅ Pre-computed {len(all_products)} product embeddings")

In [ ]:
# Load brands.json
with open('data/brands.json', 'r', encoding='utf-8') as f:
    brands_data = json.load(f)
brands = brands_data['brands']

brand_texts = [f"{b['name']} {' '.join(b.get('aliases',[]))} {b.get('note','')}" for b in brands]
brand_embeddings = emb_model.encode(brand_texts, batch_size=32, show_progress_bar=True)

np.save('models/brands_embeddings.npy', brand_embeddings)
joblib.dump(brands, 'models/brands_metadata.pkl')
print(f"✅ Pre-computed {len(brands)} brand embeddings")